In this file, we will perform the regression task, where we will create a new dataframe which will include the necessary features that will help predict the rating, a traveller/tourist gives to a particular attraction. 

In [1]:
# Importing the libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Loading the dataset
df_clean = pd.read_csv('../clean_data/df_clean.csv')

# Checking sample rows
df_clean.sample(7)

,TransactionId,UserId,VisitYear,VisitMonth,VisitModeId,AttractionId,Rating,CityId,CityName,CountryId,...,RegionId,Region,ContinentId,Continent,AttractionCityId,AttractionTypeId,Attraction,AttractionType,VisitMode,Month
8048,11732,32916,2016,1,5,640,5,3354,Kuala Lumpur,103,...,14,South East Asia,3,Asia,1,63,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Solo,January
6286,9209,64988,2017,4,4,640,4,2486,Larnaca,76,...,12,Middle East,3,Asia,1,63,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Friends,April
1433,2482,44912,2018,1,3,640,5,7468,Molsheim,159,...,21,Western Europe,5,Europe,1,63,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Family,January
43414,145330,58985,2018,1,2,369,4,535,Sydney,48,...,8,Northern America,2,America,1,13,Kuta Beach - Bali,Beaches,Couples,January
14027,21056,18051,2019,7,2,841,4,4395,Hobart,109,...,15,Australia,4,Australia & Oceania,1,92,Waterbom Bali,Water Parks,Couples,July
49102,203214,64866,2016,6,3,1171,3,3407,Shah Alam,103,...,14,South East Asia,3,Asia,3,91,Merapi Volcano,Volcanos,Family,June
41797,108704,63552,2015,6,4,737,5,3876,Bodalla,109,...,15,Australia,4,Australia & Oceania,1,76,Tanah Lot Temple,Religious Sites,Friends,June


In [3]:
# Checking the dtypes of the columns
df_clean.dtypes

TransactionId        int64
UserId               int64
VisitYear            int64
VisitMonth           int64
VisitModeId          int64
AttractionId         int64
Rating               int64
CityId               int64
CityName            object
CountryId            int64
Country             object
RegionId             int64
Region              object
ContinentId          int64
Continent           object
AttractionCityId     int64
AttractionTypeId     int64
Attraction          object
AttractionType      object
VisitMode           object
Month               object
dtype: object

In [4]:
# Creating a new dataframe with only the columns we need for the regression task
df_reg = df_clean[['Month', 'Attraction', 'AttractionType', 'VisitMode', 'Rating']].copy()

# Checking the new dataframe
df_reg.sample(7)

,Month,Attraction,AttractionType,VisitMode,Rating
21215,November,Seminyak Beach,Beaches,Couples,3
4259,October,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Friends,5
41847,April,Tanah Lot Temple,Religious Sites,Friends,5
44120,October,Kuta Beach - Bali,Beaches,Couples,3
40491,December,Tanah Lot Temple,Religious Sites,Family,5
40270,March,Tanah Lot Temple,Religious Sites,Couples,4
29530,November,Uluwatu Temple,Religious Sites,Couples,5


- Splitting the dataframe to be used as training and testing dataframes.

In [5]:
# Importing the library for train-test split
from sklearn.model_selection import train_test_split

# Splitting the data into training and testing sets
train_df, test_df = train_test_split(df_reg, test_size=0.3, random_state=42)

- Engineering features of global aggregates such as average rating of an attraction, count of ratings given to an attraction and seasonal averages of attraction types.

In [6]:
# Calculating attraction rating average and their rating count
attr_stats = train_df.groupby('Attraction')['Rating'].agg(attr_avg_rating='mean', attr_rating_count='count').reset_index()

In [7]:
# Calculating attraction type rating average
type_stats = train_df.groupby('AttractionType')['Rating'].agg(type_avg_rating='mean').reset_index()

In [8]:
# Calculating the seasonal average rating for attraction types
season_type_stats = train_df.groupby(['Month', 'AttractionType'])['Rating'].mean().reset_index(name='type_month_avg_rating')

- Merging these engineered features to the train_df and test_df.

In [9]:
# Merging the attraction stats back to the train_df and test_df dataframe
train_df = train_df.merge(attr_stats, on='Attraction', how='left')
test_df = test_df.merge(attr_stats, on='Attraction', how='left')

# Merging the attraction type stat back to the train_df and test_df dataframe
train_df = train_df.merge(type_stats, on='AttractionType', how='left')
test_df = test_df.merge(type_stats, on='AttractionType', how='left')

# Merging the seasonal average rating back to the train_df and test_df dataframe
train_df = train_df.merge(season_type_stats, on=['Month', 'AttractionType'], how='left')
test_df = test_df.merge(season_type_stats, on=['Month', 'AttractionType'], how='left')

# Calculating global mean of rating to fill in the missing values
global_mean_rating = train_df['Rating'].mean()
test_df.fillna(global_mean_rating, inplace=True)
train_df.fillna(global_mean_rating, inplace=True)

In [10]:
# Checking the updated train_df and test_df dataframe
print(train_df.sample(7))
print(test_df.sample(7))

          Month               Attraction                  AttractionType  \
11613     April            Waterbom Bali                     Water Parks   
26015    August  Tegalalang Rice Terrace  Points of Interest & Landmarks   
35492     April           Merapi Volcano                        Volcanos   
106     January           Uluwatu Temple                 Religious Sites   
22938       May            Waterbom Bali                     Water Parks   
7647   November           Merapi Volcano                        Volcanos   
32551  February           Malioboro Road           Flea & Street Markets   

      VisitMode  Rating  attr_avg_rating  attr_rating_count  type_avg_rating  \
11613   Couples       5         4.637954               4516         4.637954   
26015   Couples       4         4.170622               4003         4.140396   
35492    Family       4         4.076773               1537         4.101313   
106     Couples       5         4.212521               2348         4.1

- Separating features and targets for both train_df and test_df.

In [11]:
# Features and targets for Training data
x_train = train_df.drop('Rating', axis=1)
y_train = train_df['Rating']

# Features and targets for Testing data
x_test = test_df.drop('Rating', axis=1)
y_test = test_df['Rating']

In [12]:
# Creating a list of the categorical columns
cat_cols = ['Month', 'Attraction', 'AttractionType', 'VisitMode']

# Creating cat_features to be used later in the CatBoostRegressor model
cat_features = [x_train.columns.get_loc(col) for col in cat_cols]

In [13]:
# For LightGBM we will convert the categorical columns to category dtype
for c in cat_cols:
    x_train[c] = x_train[c].astype('category')
    x_test[c] = x_test[c].astype('category')

- Now, we will initiate an MLFlow run and train a CatBoostRegressor model.

In [14]:
# Importing the necessary libraries
import mlflow
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [15]:
# Setting the MLflow experiment name
mlflow.set_experiment('Tourism_Rating_Regression')

# Starting the MLflow run
with mlflow.start_run(run_name='CatBoostRegressor_1'):

    # Initializing the CatBoostRegressor model
    cat1 = CatBoostRegressor(iterations=300,
                              depth=6,
                              learning_rate=0.1,
                              loss_function='RMSE',
                              random_seed=42)
    
    # Fitting the model on the training data
    cat1.fit(x_train, y_train, cat_features=cat_features, eval_set=(x_test, y_test), verbose=1)
    
    # Predicting the ratings on the test data
    y_pred = cat1.predict(x_test)
    
    # Evaluating the model performance
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'CatBoostRegressor')
    mlflow.log_param('iterations', 300)
    mlflow.log_param('depth', 6)
    mlflow.log_param('learning_rate', 0.1)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # Logging the model to MLflow
    mlflow.catboost.log_model(cat1, 'cat1')

2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/24 19:21:11 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/24 19:21:13 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/24 19:21:13 INFO alembic.runtime.migration: Will assume non-transactional DDL.


0:	learn: 0.9625002	test: 0.9599413	best: 0.9599413 (0)	total: 871ms	remaining: 4m 20s
1:	learn: 0.9549539	test: 0.9525330	best: 0.9525330 (1)	total: 1.53s	remaining: 3m 47s
2:	learn: 0.9486347	test: 0.9463024	best: 0.9463024 (2)	total: 1.7s	remaining: 2m 47s
3:	learn: 0.9438327	test: 0.9414485	best: 0.9414485 (3)	total: 2.04s	remaining: 2m 31s
4:	learn: 0.9394027	test: 0.9369638	best: 0.9369638 (4)	total: 2.32s	remaining: 2m 16s
5:	learn: 0.9359139	test: 0.9335294	best: 0.9335294 (5)	total: 2.6s	remaining: 2m 7s
6:	learn: 0.9331995	test: 0.9309459	best: 0.9309459 (6)	total: 2.97s	remaining: 2m 4s
7:	learn: 0.9305280	test: 0.9283634	best: 0.9283634 (7)	total: 3.13s	remaining: 1m 54s
8:	learn: 0.9285492	test: 0.9264543	best: 0.9264543 (8)	total: 3.29s	remaining: 1m 46s
9:	learn: 0.9271339	test: 0.9250769	best: 0.9250769 (9)	total: 3.47s	remaining: 1m 40s
10:	learn: 0.9253789	test: 0.9234292	best: 0.9234292 (10)	total: 3.62s	remaining: 1m 35s
11:	learn: 0.9246640	test: 0.9227163	best: 0.

2026/02/24 19:22:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


- Cat1 model scores:

    - RMSE: 0.9145.
    - r2-score: 0.1084.

In [16]:
# Starting a second MLflow run of the same model with different parameters
with mlflow.start_run(run_name='CatBoostRegressor_2'):

    # Initializing the CatBoostRegressor model
    cat2 = CatBoostRegressor(iterations=600,
                              depth=6,
                              learning_rate=0.01,
                              loss_function='RMSE')
    
    # Fitting the model on the training data
    cat2.fit(x_train, y_train, cat_features=cat_features, eval_set=(x_test, y_test), verbose=1)
    
    # Predicting the ratings on the test data
    y_pred = cat2.predict(x_test)
    
    # Evaluating the model performance
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'CatBoostRegressor')
    mlflow.log_param('iterations', 600)
    mlflow.log_param('depth', 6)
    mlflow.log_param('learning_rate', 0.01)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # Logging the model to MLflow
    mlflow.catboost.log_model(cat2, 'cat2')

0:	learn: 0.9704930	test: 0.9677489	best: 0.9677489 (0)	total: 118ms	remaining: 1m 10s
1:	learn: 0.9695669	test: 0.9668245	best: 0.9668245 (1)	total: 230ms	remaining: 1m 8s
2:	learn: 0.9686475	test: 0.9659202	best: 0.9659202 (2)	total: 334ms	remaining: 1m 6s
3:	learn: 0.9677335	test: 0.9650243	best: 0.9650243 (3)	total: 451ms	remaining: 1m 7s
4:	learn: 0.9668603	test: 0.9641563	best: 0.9641563 (4)	total: 562ms	remaining: 1m 6s
5:	learn: 0.9660361	test: 0.9633568	best: 0.9633568 (5)	total: 690ms	remaining: 1m 8s
6:	learn: 0.9651691	test: 0.9624902	best: 0.9624902 (6)	total: 823ms	remaining: 1m 9s
7:	learn: 0.9643125	test: 0.9616641	best: 0.9616641 (7)	total: 970ms	remaining: 1m 11s
8:	learn: 0.9634998	test: 0.9608692	best: 0.9608692 (8)	total: 1.14s	remaining: 1m 14s
9:	learn: 0.9626920	test: 0.9600871	best: 0.9600871 (9)	total: 1.28s	remaining: 1m 15s
10:	learn: 0.9618787	test: 0.9592864	best: 0.9592864 (10)	total: 1.43s	remaining: 1m 16s
11:	learn: 0.9611020	test: 0.9585250	best: 0.95

2026/02/23 01:56:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


- Cat2 model scores:

    - RMSE: 0.9144.
    - r2-score: 0.1086.

- The second model with more iterations and a lower learning rate performed better than the first model. We can check the MLflow dashboard to compare the two runs and see the logged parameters and metrics.

- Next, we will do an MLFlow run using a LightGBM Regressor.

In [17]:
# Importing the libraries
from lightgbm import LGBMRegressor

In [18]:
# Starting the MLflow run
with mlflow.start_run(run_name='LGBMRegressor_1'):
    
    # Initializing the LGBMRegressor model
    light1 = LGBMRegressor(n_estimators=300,
                          max_depth=6,
                          learning_rate=0.1,
                          random_state=42)
    
    # Fitting the model on the training data
    light1.fit(x_train, y_train, eval_set=[(x_test, y_test)], categorical_feature=cat_cols)
    
    # Predicting the ratings on the test data
    y_pred = light1.predict(x_test)
    
    # Evaluating the model performance
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'LGBMRegressor')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('max_depth', 6)
    mlflow.log_param('learning_rate', 0.1)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # Logging the model to MLflow
    mlflow.lightgbm.log_model(light1, 'light1')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007362 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 37023, number of used features: 8
[LightGBM] [Info] Start training from score 4.156281
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

2026/02/23 01:56:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 01:56:59 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.


- Light1 model scores-

    - RMSE: 0.9181.
    - r2-score: 0.1015.

In [19]:
# Starting another MLflow run with different parameters
with mlflow.start_run(run_name='LGBMRegressor_2'):
    
    # Initializing the LGBMRegressor model
    light2 = LGBMRegressor(n_estimators=800,
                          max_depth=6,
                          learning_rate=0.01,
                          random_state=42)
    
    # Fitting the model on the training data
    light2.fit(x_train, y_train, eval_set=[(x_test, y_test)], categorical_feature=cat_cols)
    
    # Predicting the ratings on the test data
    y_pred = light2.predict(x_test)
    
    # Evaluating the model performance
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'LGBMRegressor')
    mlflow.log_param('n_estimators', 800)
    mlflow.log_param('max_depth', 6)
    mlflow.log_param('learning_rate', 0.01)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)

    # Logging the model to MLflow
    mlflow.lightgbm.log_model(light2, 'light2')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005152 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 37023, number of used features: 8
[LightGBM] [Info] Start training from score 4.156281
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

2026/02/23 01:57:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/23 01:57:29 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.


- Light2 model scores-

    - RMSE: 0.9163.
    - r2-score: 0.1049.

- Both the LightGBM models performed worse than the CatBoost model. So we will train a final model of CatBoost on the full dataset, which we will then save so that we can use it in Streamlit later to predict the rating.

    So for that we will compute the global aggregates on df_reg (the full dataset from which we made train_df and test_df).

In [20]:
# Calculating attraction rating average and their rating count
attr_stats_reg = df_reg.groupby('Attraction')['Rating'].agg(attr_avg_rating='mean', attr_rating_count='count').reset_index()

# Merging into df_reg
df_reg = df_reg.merge(attr_stats_reg, on='Attraction', how='left')

# Calculating attraction type rating average
type_stats_reg = df_reg.groupby('AttractionType')['Rating'].agg(type_avg_rating='mean').reset_index()

# Merging into df_reg
df_reg = df_reg.merge(type_stats_reg, on='AttractionType', how='left')

# Calculating the seasonal average rating for attraction types
season_type_stats_reg = df_reg.groupby(['Month', 'AttractionType'])['Rating'].mean().reset_index(name='type_month_avg_rating')

# Merging into df_reg
df_reg = df_reg.merge(season_type_stats_reg, on=['Month', 'AttractionType'], how='left')

# Checking the updated df_reg dataframe
df_reg.sample(7)

,Month,Attraction,AttractionType,VisitMode,Rating,attr_avg_rating,attr_rating_count,type_avg_rating,type_month_avg_rating
11462,February,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Friends,5,4.267207,13192,4.266969,4.245703
32969,October,Tegalalang Rice Terrace,Points of Interest & Landmarks,Couples,3,4.157079,5806,4.130706,4.199372
47942,June,Merapi Volcano,Volcanos,Couples,3,4.079195,2235,4.101381,4.133333
4317,October,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Couples,4,4.267207,13192,4.266969,4.276911
3696,March,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Friends,4,4.267207,13192,4.266969,4.270121
39916,September,Tanah Lot Temple,Religious Sites,Solo,5,4.195522,3350,4.207755,4.274047
50728,January,Sewu Temple,Ancient Ruins,Family,5,4.120968,372,4.055556,4.200000


In [21]:
"""Since the columns of df_reg are identical to the ones in train_df,
   the cat_features were specified using those same features, so we will not specify the categorical features again using df_reg.
   We will use the existing cat_features that was specified earlier."""

# Specifying the features and target for the regression task
x_reg = df_reg.drop('Rating', axis=1)
y_reg = df_reg['Rating']

In [ ]:
# Initializing the CatBoostRegressor model
cat_reg_final = CatBoostRegressor(iterations=600,
                              depth=6,
                              learning_rate=0.01,
                              loss_function='RMSE')
    
# Fitting the model on the training data
cat_reg_final.fit(x_reg, y_reg, cat_features=cat_features)

In [ ]:
# Saving the trained final model
import os
os.makedirs('../models', exist_ok=True)
cat_reg_final.save_model('../models/cat_reg_final.cbm')

In [ ]:
import joblib

# Storing the name of the columns of x_reg
reg_feature_cols = x_reg.columns.tolist()

# Saving the feature columns
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(reg_feature_cols, '../artifacts/reg_feature_cols.pkl')

# Saving cat_cols
joblib.dump(cat_cols, '../artifacts/reg_cat_cols.pkl')

['../artifacts/reg_cat_cols.pkl']

In [26]:
# Saving the global aggregate tables
joblib.dump(attr_stats_reg, '../artifacts/attr_stats_reg.pkl')
joblib.dump(type_stats_reg, '../artifacts/type_stats_reg.pkl')
joblib.dump(season_type_stats_reg, '../artifacts/season_type_stats_reg.pkl')

['../artifacts/season_type_stats_reg.pkl']